In [ ]:
# =====================================================================
# Grid-Integral Electromagnetic Window Optimization
# Deterministic spatial grid + analytical hotspot weights replacing
# random-trajectory Monte Carlo.  Channel model unchanged.
# =====================================================================

import torch
import numpy as np
import math
from scipy.stats import gaussian_kde
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sko.GA import GA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ==========================================
# 1. System Parameters (unchanged from MATLAB)
# ==========================================
room_x, room_y, room_z_max = 10.0, 10.0, 3.0
x_BS, y_BS, z_BS = 10.0, -100.0, -10.0

E = 8; d_B = 0.075; lambda_val = 0.075; L1 = 2
d_ref = abs(y_BS) * 1.5; P_BS_dBm = 40.0; R_th = 0.2
N0_dBm_Hz = -174.0; B = 20e6; p_m = 1.0 / 5.0; n_users = 5

P_BS = 10**(P_BS_dBm / 10.0) * 1e-3
N0 = 10**(N0_dBm_Hz / 10.0) * 1e-3 * B

# ==========================================
# ==========================================
# 2. RWP Generator (for KDE weights)
# ==========================================
def gen_rwp_traj(sim_time,dt=10,nu=5,rx=10.0,ry=10.0,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=="hot"else tau_n
    def sp(r):return v_h if r=="hot"else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]="hot"if d2h<=hr else"normal"
        if rng.rand()<p_t:us[i]="transfer";ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]="pause";uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=="pause":
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]="transfer";ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]="hot"if d2h<=hr else"normal"
                    if rng.rand()<p_s:us[i]="pause";uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

# ==========================================
# 3. Multi-Height Grid + KDE Weights
# ==========================================
GRID_RES = 200
Z_HEIGHTS = [1.5, 1.625, 1.75, 1.875, 2.0]
N_Z = len(Z_HEIGHTS)
x_vals = torch.linspace(0, room_x, GRID_RES, device=device)
y_vals = torch.linspace(0, room_y, GRID_RES, device=device)
X, Y = torch.meshgrid(x_vals, y_vals, indexing="ij")
xy_flat = torch.stack([X.flatten(), Y.flatten()], dim=1)
gl_list = []
for zh in Z_HEIGHTS:
    gl_list.append(torch.stack([X.flatten(), Y.flatten(), torch.full_like(X.flatten(), zh)], dim=1))
grid_locs = torch.cat(gl_list, dim=0)
N_GRID = grid_locs.shape[0]

print("Building KDE weights (100,000s RWP)...")
np.random.seed(777)
emp_traj = gen_rwp_traj(sim_time=100000, dt=10)
emp_xy = emp_traj[:, :2].T
kde = gaussian_kde(emp_xy, bw_method="scott")
grid_xy_np = xy_flat.cpu().numpy().T
w_xy = kde(grid_xy_np).flatten()
w_xy = w_xy / w_xy.sum()
w_full = np.tile(w_xy, N_Z)
grid_weights = torch.tensor(w_full / w_full.sum(), dtype=torch.float32, device=device)
print(f"  Grid: {N_Z} heights x {GRID_RES}x{GRID_RES} = {N_GRID} pts, hotspot/edge = {grid_weights.max().item()/grid_weights.min().item():.1f}x")

# Weight heatmap for diagnostics (use z=1.5 slice)
W_map = grid_weights[:GRID_RES*GRID_RES].reshape(GRID_RES, GRID_RES).cpu().numpy()
print(f"Weight range: [{W_map.min():.2e}, {W_map.max():.2e}]")

# ==========================================
# 4. NLoS Path-2 Parameters (fixed seed, one-time)
# ==========================================
np.random.seed(42)
_grid_nlos_psi = torch.tensor(2*np.pi*np.random.rand(N_GRID), dtype=torch.float32, device=device)
_grid_nlos_tt  = torch.tensor(-np.pi + 2*np.pi*np.random.rand(N_GRID), dtype=torch.float32, device=device)
_grid_nlos_eta = torch.tensor(10**((-15+5*np.random.rand(N_GRID))/10), dtype=torch.float32, device=device)
# 5. Channel Model — equivalent_farfield_channel_2 (unchanged)
#    LoS + NLoS, sinc diffraction, illumination gate
# ==========================================
def equivalent_farfield_channel_pytorch(window_params, locs):
    if not isinstance(window_params, torch.Tensor):
        window_params = torch.tensor(window_params, dtype=torch.float32, device=device)
    xc, zc, Lx, Lz = window_params[0], window_params[1], window_params[2], window_params[3]
    xu, yu, zu = locs[:, 0], locs[:, 1], locs[:, 2]

    dx_BS = xc - x_BS; dy_BS = torch.tensor(0.0 - y_BS, device=device); dz_BS = zc - z_BS
    R_BW = torch.sqrt(dx_BS**2 + dy_BS**2 + dz_BS**2)
    theta_BW = torch.atan2(dy_BS, dx_BS); phi_BW = torch.acos(dz_BS / R_BW)
    k_tx = torch.sin(phi_BW)*torch.cos(theta_BW); k_tz = torch.cos(phi_BW)

    x1,z1 = xc-Lx/2, zc-Lz/2; x2,z2 = xc-Lx/2, zc+Lz/2
    x3,z3 = xc+Lx/2, zc-Lz/2; x4,z4 = xc+Lx/2, zc+Lz/2

    def ray_dir_py(xs, zs):
        dx=xs-x_BS; dy=torch.tensor(0.0-y_BS,device=device); dz=zs-z_BS
        L=torch.sqrt(dx**2+dy**2+dz**2); return dx/L, dy/L, dz/L

    ux1,_,uz1=ray_dir_py(x1,z1); ux2,_,uz2=ray_dir_py(x2,z2)
    ux3,_,uz3=ray_dir_py(x3,z3); ux4,_,uz4=ray_dir_py(x4,z4)

    dx_WU=xu-x_BS; dy_WU=yu-y_BS; dz_WU=zu-z_BS
    L_USER=torch.sqrt(dx_WU**2+dy_WU**2+dz_WU**2)
    ux_user=dx_WU/L_USER; uz_user=dz_WU/L_USER

    ux_min=torch.min(torch.stack([ux1,ux2,ux3,ux4]))
    ux_max=torch.max(torch.stack([ux1,ux2,ux3,ux4]))
    uz_min=torch.min(torch.stack([uz1,uz2,uz3,uz4]))
    uz_max=torch.max(torch.stack([uz1,uz2,uz3,uz4]))

    beta=1000.0
    inx=torch.sigmoid(beta*(ux_user-ux_min))*torch.sigmoid(beta*(ux_max-ux_user))
    inz=torch.sigmoid(beta*(uz_user-uz_min))*torch.sigmoid(beta*(uz_max-uz_user))
    in_illumination=inx*inz

    dx_WU2=xu-xc; dy_WU2=yu; dz_WU2=zu-zc
    R_WU=torch.sqrt(dx_WU2**2+dy_WU2**2+dz_WU2**2)
    t1=dx_WU2/R_WU; t2=dz_WU2/R_WU
    ax=(1.0-in_illumination)*(k_tx-t1); az=(1.0-in_illumination)*(k_tz-t2)
    sincx=torch.sinc((math.pi/lambda_val)*Lx*ax)
    sincz=torch.sinc((math.pi/lambda_val)*Lz*az)

    n_ant=torch.arange(E,dtype=torch.float32,device=device)

    # Path 1 (LoS)
    phase1=(2.0*math.pi/lambda_val)*d_B*n_ant*torch.sin(theta_BW)
    a1=(1.0/math.sqrt(E))*torch.complex(torch.cos(phase1),torch.sin(phase1))
    v1_mag=torch.tensor(lambda_val/(4.0*math.pi),device=device)/R_BW
    v1_phase=torch.tensor(-(2.0*math.pi/lambda_val),device=device)*R_BW
    v1=torch.complex(v1_mag*torch.cos(v1_phase),v1_mag*torch.sin(v1_phase))
    H_path1=v1*a1.conj()

    # Path 2 (NLoS)
    psi_2,tt_2,eta_2=_grid_nlos_psi,_grid_nlos_tt,_grid_nlos_eta
    phase2=(2.0*math.pi/lambda_val)*d_B*n_ant.unsqueeze(0)*torch.sin(tt_2).unsqueeze(1)
    a2=(1.0/math.sqrt(E))*torch.complex(torch.cos(phase2),torch.sin(phase2))
    v2_mag=eta_2*(lambda_val/(4.0*math.pi*d_ref))
    v2=torch.complex(v2_mag*torch.cos(psi_2),v2_mag*torch.sin(psi_2))
    H_path2=v2.unsqueeze(1)*a2.conj()

    H_total=math.sqrt(E/L1)*(H_path1.unsqueeze(0)+H_path2)
    factor_mag=(Lx*Lz*sincx*sincz)/(lambda_val*R_WU)
    factor_phase=torch.tensor(-(2.0*math.pi/lambda_val),device=device)*(k_tx*xc+k_tz*zc)-(math.pi/2.0)
    factor=torch.complex(factor_mag*torch.cos(factor_phase),factor_mag*torch.sin(factor_phase))
    H_eq=H_total*factor.unsqueeze(1)
    return H_eq

# ==========================================
# 6. Deterministic Grid Objective
# ==========================================
penalty_weight = 500.0
target_outage = 0.10

def grid_objective(theta_input, hard_mode=True):
    if not isinstance(theta_input, torch.Tensor):
        theta_tensor = torch.tensor(theta_input, dtype=torch.float32, device=device)
    else:
        theta_tensor = theta_input

    xc,zc,Lx,Lz = theta_tensor[0],theta_tensor[1],theta_tensor[2],theta_tensor[3]
    area_raw = Lx*Lz

    H_eq = equivalent_farfield_channel_pytorch(theta_tensor, grid_locs)
    H_w = torch.sum(H_eq, dim=1)/math.sqrt(E)
    dp = (torch.abs(H_w)**2)*p_m*P_BS
    intf = (n_users-1)*dp
    sinr = dp/(intf+N0)
    rate = torch.log2(1.0+sinr)

    outage_bits = (rate < R_th).float()
    weighted_outage_val = torch.sum(outage_bits*grid_weights).item()

    if hard_mode:
        area_val = area_raw.item() if isinstance(area_raw, torch.Tensor) else float(area_raw)
        if weighted_outage_val > target_outage:
            cost = area_val + penalty_weight*(weighted_outage_val-target_outage)
        else:
            cost = area_val
        return cost, weighted_outage_val, outage_bits
    else:
        alpha_sigmoid = 300.0
        smooth_outage_vec = torch.sigmoid(alpha_sigmoid*(R_th-rate))
        smooth_outage = torch.sum(smooth_outage_vec*grid_weights)
        penalty_term = torch.relu(smooth_outage-target_outage)*penalty_weight
        return area_raw+penalty_term, weighted_outage_val, outage_bits

def boundary_penalty(x):
    xc,zc,Lx,Lz = x[0],x[1],x[2],x[3]
    v = max(0,Lx/2-xc)+max(0,xc+Lx/2-room_x)+max(0,Lz/2-zc)+max(0,zc+Lz/2-room_z_max)
    return v

def ga_obj_wrapper(x):
    cost,_,_ = grid_objective(x, hard_mode=True)
    return float(cost+boundary_penalty(x)*500.0)

# ==========================================
# 7. Warm-up Test
# ==========================================
test_x = np.array([5.0, 1.5, 0.3, 0.3])
test_cost, test_outage, _ = grid_objective(test_x)
print(f'Warm-up: window[5.0,1.5,0.3,0.3] area=0.09 m2, grid outage={test_outage*100:.2f}%\n')

# ==========================================
# 8. Phase I — GA Global Search
# ==========================================
N_GA_RUNS = 5
all_ga_results = []
_ga_eval_count = [0]

def ga_obj_wrapper_with_progress(x):
    _ga_eval_count[0] += 1
    if _ga_eval_count[0] % 500 == 0:
        print(f'    ...{_ga_eval_count[0]} evals', flush=True)
    return ga_obj_wrapper(x)

print("="*60)
print(f" Phase I: {N_GA_RUNS}-run GA  |  pop=50, gen=150")
print(f" Grid: {GRID_RES}x{GRID_RES}@z={Z_FIXED}m, analytical hotspot weights")
print("="*60)

for run_idx in range(N_GA_RUNS):
    run_seed = 20000 + run_idx*1000
    _ga_eval_count[0] = 0

    print(f"\n--- Run {run_idx+1}/{N_GA_RUNS} (seed={run_seed}) ---")
    t0 = time.time()

    ga = GA(func=ga_obj_wrapper_with_progress, n_dim=4, size_pop=50, max_iter=150,
            lb=[0.2,0.2,0.1,0.1], ub=[9.8,2.8,9.8,2.8])

    best_x_ga, best_y_ga = ga.run()
    elapsed = time.time()-t0
    ga_history = np.array(ga.generation_best_Y)  # best per generation

    _, final_outage, _ = grid_objective(best_x_ga, hard_mode=True)
    area_ga = best_x_ga[2]*best_x_ga[3]

    status = 'OK' if final_outage <= target_outage else 'WARN'
    print(f"  [{status}] area={area_ga:.4f} m2 | outage={final_outage*100:.2f}% | time={elapsed:.1f}s")
    all_ga_results.append({
        'x': best_x_ga.copy(), 'area': area_ga, 'outage': final_outage,
        'seed': run_seed, 'history': ga_history
    })

# Select the best run
valid = [r for r in all_ga_results if r['outage'] <= target_outage]
if not valid:
    valid = [r for r in all_ga_results if r['outage'] <= target_outage+0.02]
    print(f"\nNOTE: no run achieved outage < {target_outage*100:.0f}%, relaxed to < {target_outage*100+2:.0f}%")
best_result = min(valid, key=lambda r: r['area'])
print(f"\nBest GA: area={best_result['area']:.4f} m2, outage={best_result['outage']*100:.2f}%")

# ==========================================
# 9. Phase II — Adam Refinement
# ==========================================
print("\n" + "="*60)
print(" Phase II: Adam refinement (deterministic loss landscape)")
print("="*60)

refined_theta = torch.tensor(best_result['x'], dtype=torch.float32, device=device, requires_grad=True)
optimizer = torch.optim.Adam([refined_theta], lr=0.001)
adam_losses = []

for step in range(50):
    optimizer.zero_grad()
    loss, _, _ = grid_objective(refined_theta, hard_mode=False)
    loss.backward()
    adam_losses.append(loss.item())

    with torch.no_grad():
        refined_theta[2].clamp_(0.1, room_x)
        refined_theta[3].clamp_(0.1, room_z_max)
        refined_theta[0].clamp_(refined_theta[2]/2, room_x-refined_theta[2]/2)
        refined_theta[1].clamp_(refined_theta[3]/2, room_z_max-refined_theta[3]/2)
    optimizer.step()
    if step%10==0:
        print(f"  step {step:02d} | loss={loss.item():.4f}")

final_res = refined_theta.detach().cpu().numpy()
_, final_grid_outage, final_outage_bits = grid_objective(final_res, hard_mode=True)
area_final = final_res[2]*final_res[3]

# ==========================================
# 10. Results Table
# ==========================================
print("\n" + "="*72)
print("Grid-Integral EM Window Optimization")
print("="*72)
print(f" {'Parameter':<18} {'Phase I (GA)':<22} {'Phase II (Adam)':<22}")
print("-"*62)
print(f" {'Window xc [m]':<18} {best_result['x'][0]:>10.3f}            {final_res[0]:>10.3f}")
print(f" {'Window zc [m]':<18} {best_result['x'][1]:>10.3f}            {final_res[1]:>10.3f}")
print(f" {'Window Lx [m]':<18} {best_result['x'][2]:>10.3f}            {final_res[2]:>10.3f}")
print(f" {'Window Lz [m]':<18} {best_result['x'][3]:>10.3f}            {final_res[3]:>10.3f}")
print("-"*62)
print(f" {'Grid outage [%]':<18} {best_result['outage']*100:>10.2f}            {final_grid_outage*100:>10.2f}")
print(f" {'Area [m2]':<18} {best_result['area']:>10.4f}            {area_final:>10.4f}")
print("="*72)

area_dist = [f"{r['area']:.4f}" for r in all_ga_results]
print(f"\n{N_GA_RUNS}-run GA area distribution: {area_dist}")
print(f"Evaluation: KDE weights + {N_Z}-height grid, {GRID_RES}x{GRID_RES}")
print("="*72)

# ==========================================
# 11. Visualization A — Coverage Heatmap
# ==========================================
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Weight map
im1 = ax1.imshow(W_map.T, origin='lower', extent=[0,10,0,10],
                 cmap='YlOrRd', aspect='equal')
ax1.set_title('Hotspot Weight $f_{hr}(r)$', fontsize=13)
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]')
plt.colorbar(im1, ax=ax1, shrink=0.78, label='Weight density')
ax1.plot(5.0, 5.0, 'b+', markersize=14, markeredgewidth=2)

# Outage map (0=covered green, 1=outage red)
outage_map = final_outage_bits.reshape(GRID_RES, GRID_RES).cpu().numpy()
im2 = ax2.imshow(outage_map.T, origin='lower', extent=[0,10,0,10],
                 cmap='RdYlGn_r', aspect='equal', vmin=0, vmax=1)
ax2.set_title(f'Coverage Map  (outage={final_grid_outage*100:.1f}%, area={area_final:.4f} m$^2$)', fontsize=13)
ax2.set_xlabel('x [m]'); ax2.set_ylabel('y [m]')
cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.78, ticks=[0,1])
cbar2.ax.set_yticklabels(['Covered', 'Outage'])

# Draw window projection on room floor
xw1 = final_res[0]-final_res[2]/2; xw2 = final_res[0]+final_res[2]/2
ax2.plot([xw1, xw2], [0, 0], 'b-', linewidth=3, label='Window projection')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ==========================================
# 12. Visualization B — Optimization Trajectory
# ==========================================
fig2, ax = plt.subplots(figsize=(10, 4.5))

ga_hist = best_result['history']
ga_gens = np.arange(len(ga_hist))

# GA best-so-far curve
ax.plot(ga_gens, ga_hist, 'b-', linewidth=1.2, alpha=0.7, label='GA best-so-far')
ax.plot(ga_gens, ga_hist, 'b.', markersize=1.5)

# Adam steps (shifted to continue from GA endpoint)
adam_x = np.arange(len(ga_hist), len(ga_hist)+len(adam_losses))
ax.plot(adam_x, adam_losses, 'r-', linewidth=1.5, label='Adam refinement')

# Phase boundary
ax.axvline(x=len(ga_hist)-1, color='gray', linestyle='--', alpha=0.6)
ax.text(len(ga_hist)-3, ax.get_ylim()[1]*0.95, 'GA', ha='right', fontsize=10, color='blue')
ax.text(len(ga_hist)+3, ax.get_ylim()[1]*0.95, 'Adam', ha='left', fontsize=10, color='red')

ax.set_xlabel('Iteration (GA generations + Adam steps)', fontsize=11)
ax.set_ylabel('Objective (cost)', fontsize=11)
ax.set_title('Optimization Trajectory: GA Exploration + Adam Convergence', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()